# Phase 2: Stochastic Modelling & Derivatives Pricing  
## Notebook 02_11 — Heston Model Calibration

### Research question

How can a stochastic-volatility model be calibrated to an implied-volatility surface, and what diagnostics reveal whether the calibrated parameters are mathematically and financially sensible?

This notebook extends:

```text
02_09_volatility_surface_construction
```

The previous notebook built a clean implied volatility surface. This notebook fits a parametric stochastic-volatility model to option prices.

It covers:

1. Heston risk-neutral dynamics;
2. Feller condition diagnostics;
3. characteristic-function option pricing;
4. numerical integration;
5. synthetic Heston option-chain generation;
6. implied-volatility inversion;
7. calibration objective design;
8. constrained parameterisation;
9. least-squares calibration;
10. model-vs-market price and IV diagnostics;
11. smile and surface fit visualisation;
12. parameter sensitivity;
13. limitations and research-frontier extensions.

The key idea:

> Calibration is not just finding parameters. It is fitting a model, checking the fit, checking parameter plausibility, and understanding what the model cannot explain.

## 1. Why Heston?

The Black-Scholes-Merton model assumes constant volatility:

$$
dS_t=(r-q)S_tdt+\sigma S_tdW_t
$$
But option markets show volatility smiles and skews.

The Heston model replaces constant variance with a stochastic variance process:

$$
dS_t=(r-q)S_tdt+\sqrt{v_t}S_tdW_t^S
$$
$$
dv_t=\kappa(\theta-v_t)dt+\xi\sqrt{v_t}dW_t^v
$$
with instantaneous correlation:

$$
dW_t^SdW_t^v=\rho dt
$$
where:

| Parameter | Meaning |
|---|---|
| $v_0$ | initial variance |
| $\theta$ | long-run variance |
| $\kappa$ | variance mean-reversion speed |
| $\xi$ | volatility of variance, often called vol-of-vol |
| $\rho$ | spot/variance correlation |

Negative $\rho$ is one way the model generates equity-style downside skew.

## 2. Feller condition

The variance process is a CIR process.

The Feller condition is:

$$
2\kappa\theta \geq \xi^2
$$
If this holds, the variance process stays strictly positive.

If it fails, the process can hit zero, although many numerical schemes still handle this.

In real calibration, the Feller condition is often treated as a diagnostic rather than a strict hard constraint. Enforcing it can sometimes worsen the fit to market smiles.

## 3. Pricing through characteristic functions

Heston has a semi-closed-form European option price because the log-price characteristic function is available.

A European call can be written:

$$
C = S_0e^{-qT}P_1 - Ke^{-rT}P_2
$$
where $P_1$ and $P_2$ are risk-neutral probabilities computed by Fourier integration.

This notebook implements a transparent characteristic-function pricer and uses it for calibration.

Numerical warning:

> Heston pricing is sensitive to complex logarithm conventions, integration truncation, and parameter regimes. Calibration diagnostics are not optional.

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from math import erf, exp, log, sqrt
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from scipy.optimize import least_squares
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False

SCIPY_AVAILABLE

In [ ]:
@dataclass(frozen=True)
class HestonParams:
    """
    Risk-neutral Heston model parameters.

    kappa:
        Mean-reversion speed of variance.

    theta:
        Long-run variance.

    xi:
        Vol-of-vol.

    rho:
        Correlation between spot and variance Brownian motions.

    v0:
        Initial variance.
    """
    kappa: float
    theta: float
    xi: float
    rho: float
    v0: float


@dataclass(frozen=True)
class MarketConfig:
    """
    Market configuration for synthetic calibration.
    """
    spot: float = 100.0
    risk_free_rate: float = 0.035
    dividend_yield: float = 0.005
    seed: int = 42


true_params = HestonParams(
    kappa=2.0,
    theta=0.040,
    xi=0.45,
    rho=-0.65,
    v0=0.035
)

market_config = MarketConfig()

pd.Series({
    **asdict(true_params),
    **asdict(market_config)
})

In [ ]:
def feller_value(params: HestonParams) -> float:
    """
    Return 2*kappa*theta - xi^2.
    Positive means Feller condition holds.
    """
    return 2.0 * params.kappa * params.theta - params.xi ** 2


def feller_report(params: HestonParams) -> pd.Series:
    """
    Diagnostic report for the Feller condition.
    """
    value = feller_value(params)
    return pd.Series({
        "kappa": params.kappa,
        "theta": params.theta,
        "xi": params.xi,
        "rho": params.rho,
        "v0": params.v0,
        "2_kappa_theta_minus_xi_squared": value,
        "feller_condition_holds": value >= 0
    })


feller_report(true_params)

## 4. Black-Scholes helpers and implied volatility

We use Black-Scholes-Merton only as a quoting convention.

Heston generates option prices, then we invert BSM to report implied volatilities.

That is how the model-implied smile is visualised.

In [ ]:
def normal_cdf(x):
    x = np.asarray(x, dtype=float)
    return 0.5 * (1.0 + np.vectorize(erf)(x / np.sqrt(2.0)))


def bsm_price(spot, strike, maturity, r, q, vol, option_type="call"):
    spot = np.asarray(spot, dtype=float)
    if maturity <= 0:
        if option_type == "call":
            return np.maximum(spot - strike, 0.0)
        return np.maximum(strike - spot, 0.0)

    spot_safe = np.maximum(spot, 1e-12)
    d1 = (np.log(spot_safe / strike) + (r - q + 0.5 * vol**2) * maturity) / (vol * np.sqrt(maturity))
    d2 = d1 - vol * np.sqrt(maturity)

    if option_type == "call":
        return spot * np.exp(-q*maturity) * normal_cdf(d1) - strike * np.exp(-r*maturity) * normal_cdf(d2)
    if option_type == "put":
        return strike * np.exp(-r*maturity) * normal_cdf(-d2) - spot * np.exp(-q*maturity) * normal_cdf(-d1)
    raise ValueError("option_type must be call or put")


def option_bounds(spot, strike, maturity, r, q, option_type="call"):
    discounted_spot = spot * exp(-q * maturity)
    discounted_strike = strike * exp(-r * maturity)

    if option_type == "call":
        return max(discounted_spot - discounted_strike, 0.0), discounted_spot
    if option_type == "put":
        return max(discounted_strike - discounted_spot, 0.0), discounted_strike
    raise ValueError("option_type must be call or put")


def implied_vol_bisection(price, spot, strike, maturity, r, q, option_type="call", lo=1e-6, hi=4.0):
    """
    Robust BSM implied volatility by bisection.
    """
    lower, upper = option_bounds(spot, strike, maturity, r, q, option_type)
    if price < lower - 1e-8 or price > upper + 1e-8:
        return np.nan

    def objective(vol):
        return float(bsm_price(spot, strike, maturity, r, q, vol, option_type)) - price

    f_lo = objective(lo)
    f_hi = objective(hi)

    if f_lo * f_hi > 0:
        return np.nan

    for _ in range(120):
        mid = 0.5 * (lo + hi)
        f_mid = objective(mid)

        if abs(f_mid) < 1e-10 or hi - lo < 1e-10:
            return float(mid)

        if f_lo * f_mid <= 0:
            hi = mid
            f_hi = f_mid
        else:
            lo = mid
            f_lo = f_mid

    return float(0.5 * (lo + hi))

## 5. Heston characteristic function

A standard Heston characteristic function is:

$$
\phi(u)=\mathbb E[e^{iu\log S_T}]
$$
The implementation below uses a common stable form.

For $u\in\mathbb C$:

$$
\begin{aligned}
d &= \sqrt{(\rho\xi iu-\kappa)^2+\xi^2(iu+u^2)}
\end{aligned}
$$
$$
\begin{aligned}
g &= \frac{\kappa-\rho\xi iu-d}{\kappa-\rho\xi iu+d}
\end{aligned}
$$
Then:

$$
\phi(u)=\exp(C(u,T)+D(u,T)v_0+iu\log S_0)
$$
The exact form is hidden in the implementation to keep the notebook computationally focused.

In [ ]:
def heston_cf(u, params: HestonParams, maturity, spot, r, q):
    """
    Heston characteristic function for log S_T.

    This uses a common 'little Heston trap'-style formulation.
    """
    u = np.asarray(u, dtype=complex)
    i = 1j

    kappa = params.kappa
    theta = params.theta
    xi = params.xi
    rho = params.rho
    v0 = params.v0

    x0 = log(spot)
    a = kappa * theta

    d = np.sqrt((rho * xi * i * u - kappa) ** 2 + xi ** 2 * (i * u + u ** 2))
    g = (kappa - rho * xi * i * u - d) / (kappa - rho * xi * i * u + d)

    exp_neg_dT = np.exp(-d * maturity)

    C = (
        i * u * (x0 + (r - q) * maturity)
        + (a / xi ** 2)
        * (
            (kappa - rho * xi * i * u - d) * maturity
            - 2.0 * np.log((1.0 - g * exp_neg_dT) / (1.0 - g))
        )
    )

    D = (
        (kappa - rho * xi * i * u - d) / xi ** 2
        * ((1.0 - exp_neg_dT) / (1.0 - g * exp_neg_dT))
    )

    return np.exp(C + D * v0)

## 6. Heston European call price

The call price is:

$$
C=S_0e^{-qT}P_1-Ke^{-rT}P_2
$$
with:

$$
\begin{aligned}
P_1 &= \frac{1}{2} \\
&\quad + \frac{1}{\pi} \int_0^\infty \Re\left[ \frac{e^{-iu\log K}\phi(u-i)} {iu\phi(-i)} \right]du
\end{aligned}
$$
$$
\begin{aligned}
P_2 &= \frac{1}{2} \\
&\quad + \frac{1}{\pi} \int_0^\infty \Re\left[ \frac{e^{-iu\log K}\phi(u)} {iu} \right]du
\end{aligned}
$$
We approximate the integrals by trapezoidal quadrature over a finite grid.

In [ ]:
def heston_call_price(
    params: HestonParams,
    spot,
    strike,
    maturity,
    r,
    q,
    integration_limit=120.0,
    n_integration=1600
):
    """
    European call price under Heston model using characteristic function integration.
    """
    u = np.linspace(1e-8, integration_limit, n_integration)
    i = 1j
    logK = log(strike)

    phi_minus_i = heston_cf(-i, params, maturity, spot, r, q)

    phi_u = heston_cf(u, params, maturity, spot, r, q)
    phi_u_minus_i = heston_cf(u - i, params, maturity, spot, r, q)

    integrand_p1 = np.real(np.exp(-i * u * logK) * phi_u_minus_i / (i * u * phi_minus_i))
    integrand_p2 = np.real(np.exp(-i * u * logK) * phi_u / (i * u))

    p1 = 0.5 + (1.0 / np.pi) * np.trapz(integrand_p1, u)
    p2 = 0.5 + (1.0 / np.pi) * np.trapz(integrand_p2, u)

    call = spot * exp(-q * maturity) * p1 - strike * exp(-r * maturity) * p2

    lower, upper = option_bounds(spot, strike, maturity, r, q, "call")
    return float(np.clip(call, lower, upper))


def heston_put_price(params, spot, strike, maturity, r, q, **kwargs):
    """
    European put via put-call parity.
    """
    call = heston_call_price(params, spot, strike, maturity, r, q, **kwargs)
    return float(call - spot * exp(-q*maturity) + strike * exp(-r*maturity))


test_price = heston_call_price(
    true_params,
    market_config.spot,
    100.0,
    1.0,
    market_config.risk_free_rate,
    market_config.dividend_yield
)

test_iv = implied_vol_bisection(
    test_price,
    market_config.spot,
    100.0,
    1.0,
    market_config.risk_free_rate,
    market_config.dividend_yield,
    "call"
)

pd.Series({
    "heston_atm_call_1y": test_price,
    "bsm_implied_vol": test_iv
})

## 7. Synthetic market option chain

We generate a synthetic option chain from known Heston parameters, add small quote noise, and then attempt to recover the parameters by calibration.

The synthetic chain contains:

- strike;
- maturity;
- model price;
- noisy market midpoint;
- bid/ask spread;
- BSM implied volatility.

This mimics the output of a cleaned option-chain and IV pipeline.

In [ ]:
def generate_synthetic_heston_chain(
    params: HestonParams,
    market: MarketConfig,
    strikes,
    maturities,
    seed=42
):
    rng = np.random.default_rng(seed)
    rows = []

    for T in maturities:
        for K in strikes:
            model_price = heston_call_price(
                params,
                market.spot,
                float(K),
                float(T),
                market.risk_free_rate,
                market.dividend_yield,
                integration_limit=100,
                n_integration=1000
            )

            spread = max(0.015, 0.003 * model_price)
            noise = rng.normal(0.0, 0.15 * spread)
            mid = max(model_price + noise, 1e-6)
            bid = max(mid - spread / 2, 0.0)
            ask = mid + spread / 2

            iv = implied_vol_bisection(
                mid,
                market.spot,
                float(K),
                float(T),
                market.risk_free_rate,
                market.dividend_yield,
                "call"
            )

            forward = market.spot * exp((market.risk_free_rate - market.dividend_yield) * T)
            log_moneyness = log(float(K) / forward)

            rows.append({
                "strike": float(K),
                "maturity": float(T),
                "forward": forward,
                "log_moneyness": log_moneyness,
                "model_price_true": model_price,
                "bid": bid,
                "ask": ask,
                "mid": mid,
                "bid_ask_spread": ask - bid,
                "market_iv": iv,
                "true_params_kappa": params.kappa,
                "true_params_theta": params.theta,
                "true_params_xi": params.xi,
                "true_params_rho": params.rho,
                "true_params_v0": params.v0
            })

    return pd.DataFrame(rows)


strikes = np.array([70, 80, 90, 95, 100, 105, 110, 120, 130])
maturities = np.array([30, 90, 180, 365, 730]) / 365.0

market_chain = generate_synthetic_heston_chain(
    true_params,
    market_config,
    strikes,
    maturities,
    seed=market_config.seed
)

market_chain.head()

In [ ]:
plt.figure(figsize=(10, 6))

for T, group in market_chain.groupby("maturity"):
    plt.plot(group["strike"], group["market_iv"], marker="o", label=f"T={T:.2f}y")

plt.title("Synthetic Heston Implied Volatility Smiles")
plt.xlabel("Strike")
plt.ylabel("Market IV")
plt.legend()
plt.show()

## 8. Calibration objective

A calibration objective compares model prices with market prices.

A simple weighted residual is:

$$
\begin{aligned}
\epsilon_i(\Theta) &= \frac{C^{model}_i(\Theta)-C^{market}_i} {\max(\text{bid-ask spread}_i, \epsilon)}
\end{aligned}
$$
where $\Theta$ is the parameter vector:

$$
\Theta=(\kappa,\theta,\xi,\rho,v_0)
$$
We use transformed parameters so that the optimiser searches over unconstrained variables while the model parameters remain valid:

$$
\kappa=e^{x_1}
$$
$$
\theta=e^{x_2}
$$
$$
\xi=e^{x_3}
$$
$$
\rho=\tanh(x_4)
$$
$$
v_0=e^{x_5}
$$

In [ ]:
def pack_params(params: HestonParams):
    return np.array([
        log(params.kappa),
        log(params.theta),
        log(params.xi),
        np.arctanh(np.clip(params.rho, -0.999, 0.999)),
        log(params.v0)
    ], dtype=float)


def unpack_params(x):
    return HestonParams(
        kappa=float(np.exp(x[0])),
        theta=float(np.exp(x[1])),
        xi=float(np.exp(x[2])),
        rho=float(np.tanh(x[3])),
        v0=float(np.exp(x[4]))
    )


def calibration_residuals(
    x,
    chain: pd.DataFrame,
    market: MarketConfig,
    integration_limit=80,
    n_integration=800,
    feller_penalty_weight=0.0
):
    params = unpack_params(x)
    residuals = []

    for row in chain.itertuples():
        model_price = heston_call_price(
            params,
            market.spot,
            float(row.strike),
            float(row.maturity),
            market.risk_free_rate,
            market.dividend_yield,
            integration_limit=integration_limit,
            n_integration=n_integration
        )

        scale = max(float(row.bid_ask_spread), 0.03)
        residuals.append((model_price - float(row.mid)) / scale)

    if feller_penalty_weight > 0:
        violation = max(0.0, -feller_value(params))
        residuals.append(feller_penalty_weight * violation)

    return np.array(residuals, dtype=float)

## 9. Least-squares calibration

If SciPy is available, we run nonlinear least squares.

If SciPy is not available, the notebook falls back to a coarse random search.  
The random search is not meant to be production-quality; it simply keeps the notebook runnable.

In [ ]:
initial_guess = HestonParams(
    kappa=1.0,
    theta=0.030,
    xi=0.30,
    rho=-0.30,
    v0=0.030
)

x0 = pack_params(initial_guess)

if SCIPY_AVAILABLE:
    result = least_squares(
        calibration_residuals,
        x0=x0,
        args=(market_chain, market_config),
        kwargs={"integration_limit": 80, "n_integration": 700, "feller_penalty_weight": 0.0},
        max_nfev=60,
        verbose=0
    )

    calibrated_params = unpack_params(result.x)
    calibration_status = {
        "method": "scipy_least_squares",
        "success": bool(result.success),
        "message": result.message,
        "cost": float(result.cost),
        "nfev": int(result.nfev)
    }

else:
    rng = np.random.default_rng(123)
    best_x = x0
    best_score = np.inf

    for _ in range(120):
        candidate = x0 + rng.normal(0.0, [0.7, 0.5, 0.5, 0.8, 0.5])
        residual = calibration_residuals(candidate, market_chain, market_config, integration_limit=70, n_integration=500)
        score = float(np.mean(residual**2))

        if score < best_score:
            best_score = score
            best_x = candidate

    calibrated_params = unpack_params(best_x)
    calibration_status = {
        "method": "coarse_random_search_fallback",
        "success": True,
        "message": "SciPy unavailable; used coarse random search fallback.",
        "cost": best_score,
        "nfev": 120
    }

pd.Series({
    **calibration_status,
    **{f"calibrated_{k}": v for k, v in asdict(calibrated_params).items()}
})

In [ ]:
parameter_comparison = pd.DataFrame([
    {"parameter": "kappa", "true": true_params.kappa, "initial": initial_guess.kappa, "calibrated": calibrated_params.kappa},
    {"parameter": "theta", "true": true_params.theta, "initial": initial_guess.theta, "calibrated": calibrated_params.theta},
    {"parameter": "xi", "true": true_params.xi, "initial": initial_guess.xi, "calibrated": calibrated_params.xi},
    {"parameter": "rho", "true": true_params.rho, "initial": initial_guess.rho, "calibrated": calibrated_params.rho},
    {"parameter": "v0", "true": true_params.v0, "initial": initial_guess.v0, "calibrated": calibrated_params.v0},
])

parameter_comparison

In [ ]:
feller_diagnostics = pd.DataFrame([
    {"parameter_set": "true", **feller_report(true_params).to_dict()},
    {"parameter_set": "initial", **feller_report(initial_guess).to_dict()},
    {"parameter_set": "calibrated", **feller_report(calibrated_params).to_dict()},
])

feller_diagnostics

## 10. Model fit diagnostics

After calibration, we compare:

1. market price vs calibrated model price;
2. market IV vs calibrated model IV;
3. residuals by strike and maturity;
4. RMSE and MAE.

A good calibration report should diagnose where the model fits and where it fails.

In [ ]:
def build_fit_report(params: HestonParams, chain: pd.DataFrame, market: MarketConfig):
    rows = []

    for row in chain.itertuples():
        model_price = heston_call_price(
            params,
            market.spot,
            float(row.strike),
            float(row.maturity),
            market.risk_free_rate,
            market.dividend_yield,
            integration_limit=100,
            n_integration=1000
        )

        model_iv = implied_vol_bisection(
            model_price,
            market.spot,
            float(row.strike),
            float(row.maturity),
            market.risk_free_rate,
            market.dividend_yield,
            "call"
        )

        rows.append({
            "strike": float(row.strike),
            "maturity": float(row.maturity),
            "log_moneyness": float(row.log_moneyness),
            "market_mid": float(row.mid),
            "model_price": model_price,
            "price_error": model_price - float(row.mid),
            "abs_price_error": abs(model_price - float(row.mid)),
            "market_iv": float(row.market_iv),
            "model_iv": model_iv,
            "iv_error": model_iv - float(row.market_iv),
            "abs_iv_error": abs(model_iv - float(row.market_iv)),
            "bid_ask_spread": float(row.bid_ask_spread)
        })

    return pd.DataFrame(rows)


fit_report = build_fit_report(calibrated_params, market_chain, market_config)

fit_summary = pd.Series({
    "price_rmse": float(np.sqrt(np.mean(fit_report["price_error"]**2))),
    "price_mae": float(np.mean(fit_report["abs_price_error"])),
    "iv_rmse": float(np.sqrt(np.mean(fit_report["iv_error"]**2))),
    "iv_mae": float(np.mean(fit_report["abs_iv_error"])),
    "max_abs_iv_error": float(np.max(fit_report["abs_iv_error"]))
})

fit_summary

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(fit_report["market_mid"], fit_report["model_price"], alpha=0.8)
min_p = min(fit_report["market_mid"].min(), fit_report["model_price"].min())
max_p = max(fit_report["market_mid"].max(), fit_report["model_price"].max())
plt.plot([min_p, max_p], [min_p, max_p], linestyle="--")
plt.title("Heston Calibration: Market Price vs Model Price")
plt.xlabel("Market midpoint")
plt.ylabel("Calibrated model price")
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(fit_report["market_iv"], fit_report["model_iv"], alpha=0.8)
min_v = min(fit_report["market_iv"].min(), fit_report["model_iv"].min())
max_v = max(fit_report["market_iv"].max(), fit_report["model_iv"].max())
plt.plot([min_v, max_v], [min_v, max_v], linestyle="--")
plt.title("Heston Calibration: Market IV vs Model IV")
plt.xlabel("Market IV")
plt.ylabel("Model IV")
plt.show()

## 11. Smile fit by maturity

A calibration may fit the average surface but miss specific maturity slices.

We plot market IV and calibrated Heston IV by maturity.

In [ ]:
plt.figure(figsize=(11, 6))

for T, group in fit_report.groupby("maturity"):
    group = group.sort_values("strike")
    plt.plot(group["strike"], group["market_iv"], marker="o", linestyle="--", alpha=0.7, label=f"Market T={T:.2f}")
    plt.plot(group["strike"], group["model_iv"], marker="x", alpha=0.7, label=f"Heston T={T:.2f}")

plt.title("Market vs Calibrated Heston Smiles")
plt.xlabel("Strike")
plt.ylabel("BSM implied volatility")
plt.legend(ncol=2, fontsize=8)
plt.show()

In [ ]:
residual_pivot = fit_report.pivot(index="maturity", columns="strike", values="iv_error")

plt.figure(figsize=(11, 5))
plt.imshow(
    residual_pivot.values,
    origin="lower",
    aspect="auto",
    extent=[
        residual_pivot.columns.min(),
        residual_pivot.columns.max(),
        residual_pivot.index.min(),
        residual_pivot.index.max()
    ]
)
plt.colorbar(label="Model IV - Market IV")
plt.title("Heston Calibration IV Residual Heatmap")
plt.xlabel("Strike")
plt.ylabel("Maturity")
plt.show()

## 12. Parameter sensitivity: skew and vol-of-vol

Heston's smile shape is strongly influenced by:

- $\rho$, which controls spot/variance correlation and skew;
- $\xi$, which controls vol-of-vol and smile curvature.

We perturb these parameters and inspect their effect on a one-year smile.

In [ ]:
def heston_iv_smile(params, market, strikes, maturity):
    rows = []
    for K in strikes:
        price = heston_call_price(
            params,
            market.spot,
            float(K),
            maturity,
            market.risk_free_rate,
            market.dividend_yield,
            integration_limit=100,
            n_integration=900
        )
        iv = implied_vol_bisection(
            price,
            market.spot,
            float(K),
            maturity,
            market.risk_free_rate,
            market.dividend_yield,
            "call"
        )
        rows.append({"strike": float(K), "maturity": maturity, "price": price, "iv": iv})
    return pd.DataFrame(rows)


rho_scenarios = [-0.85, -0.50, -0.10, 0.30]
rho_rows = []

for rho_value in rho_scenarios:
    p = HestonParams(
        kappa=calibrated_params.kappa,
        theta=calibrated_params.theta,
        xi=calibrated_params.xi,
        rho=rho_value,
        v0=calibrated_params.v0
    )
    smile = heston_iv_smile(p, market_config, strikes, maturity=1.0)
    smile["rho"] = rho_value
    rho_rows.append(smile)

rho_sensitivity = pd.concat(rho_rows, ignore_index=True)

plt.figure(figsize=(10, 6))
for rho_value, group in rho_sensitivity.groupby("rho"):
    plt.plot(group["strike"], group["iv"], marker="o", label=f"rho={rho_value}")
plt.title("Heston Smile Sensitivity to Correlation rho")
plt.xlabel("Strike")
plt.ylabel("Implied volatility")
plt.legend()
plt.show()

In [ ]:
xi_scenarios = [0.20, 0.35, 0.55, 0.80]
xi_rows = []

for xi_value in xi_scenarios:
    p = HestonParams(
        kappa=calibrated_params.kappa,
        theta=calibrated_params.theta,
        xi=xi_value,
        rho=calibrated_params.rho,
        v0=calibrated_params.v0
    )
    smile = heston_iv_smile(p, market_config, strikes, maturity=1.0)
    smile["xi"] = xi_value
    xi_rows.append(smile)

xi_sensitivity = pd.concat(xi_rows, ignore_index=True)

plt.figure(figsize=(10, 6))
for xi_value, group in xi_sensitivity.groupby("xi"):
    plt.plot(group["strike"], group["iv"], marker="o", label=f"xi={xi_value}")
plt.title("Heston Smile Sensitivity to Vol-of-Vol xi")
plt.xlabel("Strike")
plt.ylabel("Implied volatility")
plt.legend()
plt.show()

## 13. Calibration hygiene checklist

A Heston calibration should report:

1. optimiser status;
2. objective value;
3. parameter bounds or transforms;
4. Feller diagnostic;
5. price errors;
6. implied-volatility errors;
7. residuals by maturity and strike;
8. sensitivity to starting guesses;
9. integration settings;
10. parameter plausibility.

Without these diagnostics, a calibration result is just five numbers.

## 14. Saving calibration outputs

The notebook saves:

1. synthetic market option chain;
2. calibration status;
3. parameter comparison;
4. Feller diagnostics;
5. fit report;
6. fit summary;
7. rho sensitivity;
8. xi sensitivity;
9. manifest.

In [ ]:
output_dir = Path("data/processed/heston_model_calibration_v1")
output_dir.mkdir(parents=True, exist_ok=True)

market_chain_path = output_dir / "synthetic_heston_market_chain.csv"
status_path = output_dir / "calibration_status.json"
params_path = output_dir / "parameter_comparison.csv"
feller_path = output_dir / "feller_diagnostics.csv"
fit_report_path = output_dir / "fit_report.csv"
fit_summary_path = output_dir / "fit_summary.csv"
rho_sensitivity_path = output_dir / "rho_sensitivity.csv"
xi_sensitivity_path = output_dir / "xi_sensitivity.csv"
manifest_path = output_dir / "manifest.json"

market_chain.to_csv(market_chain_path, index=False)
parameter_comparison.to_csv(params_path, index=False)
feller_diagnostics.to_csv(feller_path, index=False)
fit_report.to_csv(fit_report_path, index=False)
fit_summary.to_frame("value").to_csv(fit_summary_path)
rho_sensitivity.to_csv(rho_sensitivity_path, index=False)
xi_sensitivity.to_csv(xi_sensitivity_path, index=False)

status_path.write_text(json.dumps(calibration_status, indent=2, default=str))

manifest = {
    "dataset_name": "heston_model_calibration_outputs",
    "schema_version": "heston_model_calibration_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "market_config": asdict(market_config),
    "true_params": asdict(true_params),
    "initial_guess": asdict(initial_guess),
    "calibrated_params": asdict(calibrated_params),
    "calibration_status": calibration_status,
    "feller_value_calibrated": float(feller_value(calibrated_params)),
    "feller_holds_calibrated": bool(feller_value(calibrated_params) >= 0),
    "fit_summary": fit_summary.to_dict(),
    "notes": [
        "Synthetic market prices are generated from known Heston parameters with small quote noise.",
        "Calibration objective uses bid-ask scaled price residuals.",
        "Parameters are transformed to enforce positivity and rho in (-1,1).",
        "Feller condition is reported as a diagnostic, not imposed as a hard constraint.",
        "Characteristic-function pricing uses finite trapezoidal integration."
    ]
}

manifest_path.write_text(json.dumps(manifest, indent=2, default=str))

market_chain_path, params_path, fit_report_path, manifest_path

## 15. Limitations

### 15.1 Synthetic data

This notebook calibrates to synthetic Heston prices.

Real market calibration requires:

- bid/ask filtering;
- dividend/forward curves;
- discount curves;
- stale quote removal;
- static-arbitrage checks;
- robust surface preprocessing.

### 15.2 Numerical integration sensitivity

Heston prices depend on:

- integration truncation;
- grid resolution;
- characteristic-function convention;
- complex logarithm stability.

Production pricers often use carefully tested Fourier methods.

### 15.3 Parameter non-uniqueness

Different parameter sets can generate similar option prices.

This is especially true when the option chain is sparse or noisy.

### 15.4 Feller condition

The Feller condition is not imposed.

This is realistic for many calibrations, but the consequence is that variance may hit zero under the model.

### 15.5 One-day calibration

This notebook calibrates one surface snapshot.

Real systems monitor parameter stability over time.

### 15.6 Heston may not fit all smiles

Heston can generate skew and curvature, but may struggle with:

- very short maturities;
- steep wings;
- jumps;
- rough volatility behaviour;
- event-driven smiles.

### 15.7 No Greeks or hedging yet

Calibration alone does not prove hedging performance.

Model Greeks and hedging errors should be tested separately.

## 16. Research frontier and current directions

### 16.1 Bates model

The Bates model adds jumps to Heston stochastic volatility.

This helps capture both volatility clustering and discontinuous jump risk.

### 16.2 Rough Heston

Rough volatility models replace smooth variance dynamics with rougher paths.

They can fit short-maturity smile behaviour better than classical Heston in many markets.

### 16.3 Fourier acceleration

Carr-Madan FFT, COS methods, and other transform methods can price large strike grids efficiently.

A future notebook could compare trapezoidal integration with COS pricing.

### 16.4 Calibration regularisation

Production calibration often penalises unstable parameters or large day-to-day parameter changes.

This improves parameter stability but introduces model governance choices.

### 16.5 Joint calibration across products

Equity index, single-name, futures, rates, and FX options may require different conventions.

For Chinese futures options, Black-76-style forwards and product-specific calendars become important.

### 16.6 Neural calibration

Neural networks can learn the inverse map from surfaces to model parameters or approximate prices directly.

The challenge is enforcing no-arbitrage and preserving interpretability.

## 17. Suggested follow-up notebooks

This notebook naturally leads to:

1. **02_12_sabr_smile_calibration**  
   Smile calibration with a different parametric model.

2. **02_14_local_volatility_dupire_surface**  
   Converting a surface into local volatility.

3. **02_Bates_stochastic_volatility_with_jumps**  
   Adding jumps to stochastic volatility.

4. **02_13_multilevel_monte_carlo_pricing**  
   Simulating stochastic volatility paths efficiently.

5. **02_08_dynamic_delta_hedging_simulation**  
   Testing hedging performance under stochastic volatility.

6. **03_09_volatility_surface_alpha_signals**  
   Turning surface features into research signals.

7. **07_02_volatility_surface_pricing_and_alpha**  
   Capstone linking surface construction, calibration, and alpha.

## 18. Summary

This notebook implemented Heston model calibration.

It showed how to:

1. define Heston risk-neutral dynamics;
2. compute the Feller condition;
3. price European calls using the characteristic function;
4. generate a synthetic Heston option chain;
5. invert prices into BSM implied volatilities;
6. define a bid-ask-scaled calibration objective;
7. calibrate transformed parameters;
8. compare true, initial, and calibrated parameters;
9. diagnose model fit in price and IV space;
10. study smile sensitivity to $\rho$ and $\xi$;
11. save calibration reports and metadata.

The key computational takeaway:

> A calibration pipeline is an optimisation system with numerical integration, constraints, diagnostics, and error reporting.

The key financial takeaway:

> Heston turns a volatility surface into interpretable stochastic-volatility parameters, but the fit must be judged by residuals, stability, and model limitations.

## 19. Further reading

- Heston, S. L. "A Closed-Form Solution for Options with Stochastic Volatility with Applications to Bond and Currency Options."
- Gatheral, J. *The Volatility Surface: A Practitioner's Guide.*
- Andersen and Piterbarg, *Interest Rate Modeling*, volatility modelling sections.
- Lewis, A. *Option Valuation under Stochastic Volatility.*
- Lord and Kahl on Heston characteristic-function formulations.
- Fang and Oosterlee on COS methods.
- Bates stochastic volatility with jumps.
- Rough Heston and rough volatility literature.